In [7]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Ensure the notebook can find the /src directory
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from src.data_loader import VesselDataLoader
from src.visualizer import plot_block_space, plot_mode_statistics
from src.data_processing import engineer_telemetry_features
from src.mission_profiler import MissionProfiler

In [12]:
filename1 = "Rotherhithe_voy_179.csv"
filename2 = "Wembley_voy_236.csv"

filename = filename1 

raw_data_file = root_path / "data" / "raw" / filename

loader = VesselDataLoader(raw_data_file)
raw_df = loader.load_and_clean()

processed_df = engineer_telemetry_features(raw_df, filter_method='savgol')

profiler = MissionProfiler(speed_threshold = 0.01)

modes_df = profiler.classify_modes(processed_df)
global_stats = profiler.extract_global_statistics(modes_df)

registry_df = profiler.generate_block_registry(modes_df, source_file_name=filename, merge_loitering=True, unify_port_ops=False)
display(registry_df.head())

,Source_File,Start_Time,Duration_h,Energy_kWh,Mean_Power_kW,H2_Rate_Lower_kg_h,H2_Rate_Upper_kg_h,Relative_Fatigue_Activity_Rate,Mean_Power_Fluctuation_Intensity,Stay_ID,MODE,Loitering_Handling,Port_Handling
0,Rotherhithe_voy_179.csv,2025-12-22 10:50,103.666667,42999.118700,414.782495,22.633553,27.663232,0.137294,5.196725,2,Sea_Transit_Ballast,Merged,NaN
1,Rotherhithe_voy_179.csv,2025-12-27 15:55,5.250000,2414.215185,459.850511,25.092792,30.668968,0.050038,2.184905,3,Port_Idle,NaN,Separated
2,Rotherhithe_voy_179.csv,2025-12-26 18:30,21.416667,10237.224212,478.002687,26.083307,31.879598,0.115101,4.363502,3,Port_Loading,NaN,Separated
3,Rotherhithe_voy_179.csv,2025-12-27 21:10,60.416667,29626.396332,490.367939,26.758045,32.704278,0.015265,0.603777,4,Sea_Transit_Laden,Merged,NaN
4,Rotherhithe_voy_179.csv,2025-12-30 09:35,2.500000,1438.260178,575.304071,31.392779,38.368952,0.615356,19.732394,5,Port_Idle,NaN,Separated


In [13]:
fig_bricks_fluctuation = plot_block_space(registry_df, y_axis_metric='Mean_Power_Fluctuation_Intensity')
fig_bricks_fluctuation.show()
fig_bricks_fatigue = plot_block_space(registry_df, y_axis_metric='Relative_Fatigue_Activity_Rate')
fig_bricks_fatigue.show()


In [11]:
fig_stats_fluctuation = plot_mode_statistics(global_stats, y_axis_metric='Mean_Power_Fluctuation_Intensity')
fig_stats_fluctuation.show()

fig_stats_fatigue = plot_mode_statistics(global_stats, y_axis_metric='Relative_Fatigue_Activity_Rate')
fig_stats_fatigue.show()